# User Referral Program

This notebook reproduces the cleaned referral-program analysis used to create the final PDF report.

The workflow:

1. Load the transaction data.
2. Validate the referral launch date and referral flags.
3. Clean inconsistent user-level referral labels.
4. Calculate before/after impact metrics.
5. Analyze referral contribution, incentive economics, device/fraud signals, and country-level performance.
6. Generate the cleaned PDF report.


In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/Users/jerrywyatt/Documents/Codex/2026-07-08/12-16-user-referral-program-goal")
DATA_PATH = Path("/Users/jerrywyatt/Downloads/referral.csv")
OUTPUT_DIR = BASE_DIR / "outputs"
LAUNCH_DATE = pd.Timestamp("2015-10-31")

df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df.head()

In [ ]:
df.shape, df["date"].min(), df["date"].max(), df["date"].nunique()

## Data Quality Check

In [ ]:
# Referral program started on 2015-10-31, so there should be no referral transactions before launch.
pd.crosstab(df["date"] >= LAUNCH_DATE, df["is_referral"], rownames=["post_launch"], colnames=["is_referral"])

In [ ]:
# Users should not normally appear as both referred and non-referred.
mixed_users = (df.groupby("user_id")["is_referral"].nunique() > 1).sum()
mixed_users

In [ ]:
# Match the provided solution's cleanup: assign each user their first observed referral status.
df["is_referral_fixed"] = df.groupby("user_id")["is_referral"].transform("first")
df["period"] = df["date"].apply(lambda d: "Post-launch" if d >= LAUNCH_DATE else "Pre-launch")

df.groupby("user_id")["is_referral_fixed"].nunique().max()

## Before vs. After Launch

In [ ]:
pre = df[df["date"] < LAUNCH_DATE]
post = df[df["date"] >= LAUNCH_DATE]

summary = pd.DataFrame([
    {
        "period": "Before launch",
        "transactions": len(pre),
        "unique_users": pre["user_id"].nunique(),
        "unique_devices": pre["device_id"].nunique(),
        "revenue": pre["money_spent"].sum(),
        "aov": pre["money_spent"].mean(),
    },
    {
        "period": "After launch",
        "transactions": len(post),
        "unique_users": post["user_id"].nunique(),
        "unique_devices": post["device_id"].nunique(),
        "revenue": post["money_spent"].sum(),
        "aov": post["money_spent"].mean(),
    },
])

summary

In [ ]:
daily_pre_revenue = pre.groupby("date")["money_spent"].sum().mean()
daily_post_revenue = post.groupby("date")["money_spent"].sum().mean()
daily_pre_users = pre.groupby("date")["user_id"].nunique().mean()
daily_post_users = post.groupby("date")["user_id"].nunique().mean()

impact = pd.DataFrame({
    "metric": ["Average daily revenue", "Average daily purchasing users"],
    "before_launch": [daily_pre_revenue, daily_pre_users],
    "after_launch": [daily_post_revenue, daily_post_users],
})
impact["pct_change"] = impact["after_launch"] / impact["before_launch"] - 1
impact

## Referral Contribution

In [ ]:
post_segments = (
    post.groupby("is_referral_fixed")
    .agg(
        transactions=("user_id", "size"),
        users=("user_id", "nunique"),
        revenue=("money_spent", "sum"),
        devices=("device_id", "nunique"),
        aov=("money_spent", "mean"),
    )
    .rename(index={0: "Non-referral", 1: "Referral"})
)

post_segments

## Incentive Economics

In [ ]:
referred_users = post_segments.loc["Referral", "users"]
referral_revenue = post_segments.loc["Referral", "revenue"]
incremental_revenue = post["money_spent"].sum() - pre["money_spent"].sum()
estimated_credit_cost = referred_users * 10

economics = pd.DataFrame({
    "metric": [
        "Cleaned referred users",
        "Estimated credit cost at $10 per referred user",
        "Referral revenue",
        "Credit cost / referral revenue",
        "Credit cost / incremental revenue",
    ],
    "value": [
        referred_users,
        estimated_credit_cost,
        referral_revenue,
        estimated_credit_cost / referral_revenue,
        estimated_credit_cost / incremental_revenue,
    ],
})

economics

## Device / Fraud Signal

In [ ]:
device_rows = []
for label, g in [("Before launch", pre), ("After launch", post)]:
    users_per_device = g.groupby("device_id")["user_id"].nunique()
    device_rows.append({
        "period": label,
        "avg_daily_users": g.groupby("date")["user_id"].nunique().mean(),
        "avg_daily_devices": g.groupby("date")["device_id"].nunique().mean(),
        "users_per_device": users_per_device.mean(),
        "multi_user_device_share": (users_per_device > 1).mean(),
        "max_users_per_device": users_per_device.max(),
    })

device_summary = pd.DataFrame(device_rows)
device_summary

## Country-Level Readout

In [ ]:
country_rows = []
for country_code in sorted(df["country"].unique()):
    a = pre[pre["country"] == country_code]
    b = post[post["country"] == country_code]
    country_rows.append({
        "country": country_code,
        "revenue_change": b["money_spent"].sum() - a["money_spent"].sum(),
        "transaction_change": len(b) - len(a),
        "referral_revenue": b.loc[b["is_referral_fixed"] == 1, "money_spent"].sum(),
    })

country_summary = pd.DataFrame(country_rows).sort_values("revenue_change", ascending=False)
country_summary

## Generate the PDF Report

In [ ]:
# The report builder contains the formatted PDF layout, embedded line charts, and final recommendations.
# Running this cell regenerates outputs/clean_user_referral_program_solution.pdf.
import subprocess
import sys

builder = BASE_DIR / "work" / "build_clean_referral_report.py"
result = subprocess.run([sys.executable, str(builder)], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Final Recommendation

The referral program appears promising because users and revenue increased after launch, and referred users contributed substantial post-launch revenue. However, the data also shows signs of cannibalization and possible abuse. The program should continue with stronger guardrails, anti-fraud checks, and a controlled experiment before scaling incentives or changing reward levels.
